In [125]:
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator
import math
from pathlib import Path

from xarch_tokenizers.logging.report_utils import (
    load_predictions,
    load_all_samples,
    clean_model_name,
)
from xarch_tokenizers.logging.plot_utils import (
    setup_styles,
    get_new_6_fig,
    MODEL_TO_COLOR,
    get_new_fig,
)


setup_styles()
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [126]:
def get_canonical_and_perturbed_df(
    samples,
    only_keep_canonical_true: bool = False,
    keep_remains_or_becomes_correct: bool = False,
    only_keep_perturbed_true: bool = False,
    verbose: bool = False,
):
    """returns filtered out samples too"""
    if (
        (only_keep_canonical_true and keep_remains_or_becomes_correct)
        or (only_keep_canonical_true and only_keep_perturbed_true)
        or (keep_remains_or_becomes_correct and only_keep_perturbed_true)
    ):
        raise ValueError(
            f"Please pass these exclusively, only_keep_canonical_true: true for canonical, model_name true, keep_remains_or_becomes_correct: True to include the ones where the perturbed version becomes correct"
        )
    if verbose:
        print(f"Processing {len(samples)} examples")
    canonical_samples = samples[samples["is_canonical"]]
    perturbed_samples = samples[~samples["is_canonical"]]
    canonical_values = canonical_samples[["set_id", "model_name", METRIC]].rename(
        columns={METRIC: f"canonical_{METRIC}"}
    )

    perturbed_samples = perturbed_samples.merge(
        canonical_values, on=["set_id", "model_name"], how="left"
    )
    perturbed_samples[f"canonical-perturbed_{METRIC}"] = (
        perturbed_samples[f"canonical_{METRIC}"] - perturbed_samples[METRIC]
    )

    if only_keep_canonical_true:
        correct_canonical_pairs = canonical_samples[canonical_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ]
        if verbose:
            print(
                f"Cleaning samples, will be dropping entries (set_id-model pairs) where the canonical is predicted wrong."
            )
            print(
                f"Keeping {len(correct_canonical_pairs)} correct pairs, dropping {len(canonical_samples) - len(correct_canonical_pairs)} entries."
            )

        canonical_samples = canonical_samples[canonical_samples[METRIC] == 1]

        # Step 3: Filter perturbed_samples to keep only rows with correct canonical pairs
        # Method 1: Using merge (RECOMMENDED)
        perturbed_samples = perturbed_samples.merge(
            correct_canonical_pairs,
            on=["set_id", "model_name"],
            how="inner",  # Only keep rows that exist in both
        )
        samples = pd.concat((canonical_samples, perturbed_samples), ignore_index=True)

    if keep_remains_or_becomes_correct:
        # Get pairs where perturbed samples are correct (becomes or remains correct)
        correct_perturbed_pairs = perturbed_samples[perturbed_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ].drop_duplicates()

        # Get pairs where canonical samples are correct (remains correct)
        correct_canonical_pairs = canonical_samples[canonical_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ]

        # Combine: pairs that either remain correct OR become correct
        remains_or_becomes_correct = pd.concat(
            [correct_canonical_pairs, correct_perturbed_pairs]
        ).drop_duplicates()

        # Filter both datasets to only include these pairs
        canonical_samples = canonical_samples.merge(
            remains_or_becomes_correct, on=["set_id", "model_name"], how="inner"
        )

        perturbed_samples = perturbed_samples.merge(
            remains_or_becomes_correct, on=["set_id", "model_name"], how="inner"
        )
        samples = pd.concat((canonical_samples, perturbed_samples), ignore_index=True)
        if verbose:
            print(f"After filtering:{len(samples)}")
    if only_keep_perturbed_true:
        # Get pairs where perturbed samples are correct (becomes or remains correct)
        correct_perturbed_pairs = perturbed_samples[perturbed_samples[METRIC] == 1][
            ["set_id", "model_name"]
        ].drop_duplicates()

        # Get pairs where canonical samples are correct (remains correct)
        wrong_canonical_pairs = canonical_samples[canonical_samples[METRIC] == 0][
            ["set_id", "model_name"]
        ]
        # Find cases where canonical wrong AND perturbed correct (recovery cases)
        becomes_correct = pd.merge(
            wrong_canonical_pairs,
            correct_perturbed_pairs,
            how="inner",
            on=["set_id", "model_name"],
        )

        # Filter both datasets to only include recovery cases
        canonical_samples = canonical_samples.merge(
            becomes_correct, on=["set_id", "model_name"], how="inner"
        )

        perturbed_samples = perturbed_samples.merge(
            becomes_correct, on=["set_id", "model_name"], how="inner"
        )

        samples = pd.concat([canonical_samples, perturbed_samples], ignore_index=True)
        if verbose:
            print(f"After filtering: {len(samples)}")

    return samples, canonical_samples, perturbed_samples

# LEt's Create Summary

In [ ]:
OUTPUT_DIR = Path("./output/results-v5")
OUTPUT_DIR_ = OUTPUT_DIR / "summary"
METRIC = "acc_norm"
# read all samples
base_dir = Path("../results/paper-v5").resolve().absolute()

# # keeps set_id, model pairs that are correct for canonical
# suffix = "(Still correct)"
# only_keep_canonical_true = True
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = False

# suffix = "(Remains or Becomes Correct)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = True
# only_keep_perturbed_true = False

# suffix = "(Flipped to correct)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = True

# suffix = "(All)"
# only_keep_canonical_true = False
# keep_remains_or_becomes_correct = False
# only_keep_perturbed_true = False

code_pattern = "*"
title_pattern = "All Datasets"
OUTPUT_DIR = OUTPUT_DIR_ / "all"

OUTPUT_DIR = OUTPUT_DIR 
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
base_dir.exists()
pred_files = list([p for p in base_dir.rglob(f"samples{code_pattern}.jsonl")])
len(pred_files), pred_files[:5]


(2234,
 [PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_general_currency_symbol_2025-09-20T10-09-51.814046.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_chinese_ocr_errors_2025-09-20T11-23-04.491388.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_typographical_errors_2025-09-20T10-31-24.441216.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_grammatical_errors_2025-09-20T10-31-24.441216.jsonl'),
  PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-

In [174]:
## Read all samples into a DataFrame
## exclude general dataset for now
all_samples = load_all_samples(
    base_dir,
    patterns=[code_pattern],
    exclude_patterns=["general"],
    flatten_doc=True,
    match_date=False,
    simplify_df=True,
)

2072 [PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_chinese_ocr_errors_2025-09-20T11-23-04.491388.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_typographical_errors_2025-09-20T10-31-24.441216.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_italian_grammatical_errors_2025-09-20T10-31-24.441216.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samples_tokenizer_robustness_completion_english_contractions_2025-09-20T00-37-11.241572.jsonl'), PosixPath('/Users/gsaltintas/code/phd/tokenizers/results/paper-v5/r-three__supertoken_models-llama_facebook-xglm-564M/samp

We want a summary table like below

| filtering_mode   | model_name   | task                                                      | task_pretty_name     |   canonical_count |   num_samples | category                 | langs    |   acc_norm |   acc_norm_std |      acc |   acc_std | canonical_task_name                            |   canonical_acc |   canonical_acc_norm |
|:-----------------|:-------------|:----------------------------------------------------------|:---------------------|------------------:|--------------:|:-------------------------|:---------|-----------:|---------------:|---------:|----------:|:-----------------------------------------------|----------------:|---------------------:|
| no_filter        | Aya          | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.352941 |  0.492592 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | BLOOM        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | ByT5         | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Comma        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | GPT-2        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.235294 |  0.437237 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.823529 |
| no_filter        | GPT-4o       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.352941 |  0.492592 | tokenizer_robustness_completion_stem_canonical |        1        |             0.941176 |
| no_filter        | Gemma-2      | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.882353 |
| no_filter        | Llama-3.2    | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.352941 |       0.492592 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.882353 |
| no_filter        | Phi-3        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.411765 |       0.5073   | 0.235294 |  0.437237 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Qwen-3       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | Tekken       | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.235294 |       0.437237 | 0.176471 |  0.392953 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.882353 |
| no_filter        | TokenMonster | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.941176 |             0.764706 |
| no_filter        | XGLM         | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.823529 |       0.392953 | 0.882353 |  0.332106 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.823529 |
| no_filter        | mBERT        | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.235294 |       0.437237 | 0.294118 |  0.469668 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.764706 |


started with this

|          |                     |            |     |          |                   |                     |             |                  |                     |                     |
| -------- | ------------------- | ---------- | --- | -------- | ----------------- | ------------------- | ----------- | ---------------- | ------------------- | ------------------- |
| Category | Perturbation (Task) | Model name | acc | acc_norm | number of samples | filter              | acc_std_err | acc_norm_std_err | number of canonical | number of perturbed |
|          | romanization        | Comma      |     |          |                   | only_canonical_true |             |                  |                     |                     |
|          | dialect             | Comma      |     |          |                   | only_canonical_true |             |                  |                     |                     |
|          | romanization        | Comma      |     |          |                   | no_filter           |             |                  |                     |                     |
|          | dialect             | Comma      |     |          |                   | no_filter           |             |                  |                     |                     |
|          |                     |            |     |          |                   | only_canonical      |             |                  |                     |                     |
|          |                     |            |     |          |                   | only_perturbed      |             |                  |                     |                     |								

In [ ]:
import warnings

from category_mapping import SUBCATEGORY_TO_CATEGORY

all_tasks = all_samples["task_pretty_name"].unique()
all_tasks = all_samples["task"].unique()
model_names = all_samples["model_name"].unique()
## cleanup secondary categories
all_samples["final_category"] = all_samples["category"].apply(
    lambda x: str(x).split(",")[0]
)

all_summaries = []

FILTERS = [
    "only_canonical_correct",
    "no_filter",
    "canonical_wrong_at_least_one_perturb_correct",
    "remains_or_becomes_correct"
]
keys = {
    "only_canonical_correct": (True, False, False),
    "no_filter": (False, False, False),
    "canonical_wrong_at_least_one_perturb_correct": (False, False, True),
    "remains_or_becomes_correct": (False, True, False),
}

for filtering_mode in FILTERS:
    #################### filtering & clean-up ####################
    # don't rely on canonical_df, perturbed_df, always use samples
    samples, canonical_df, perturbed_df = get_canonical_and_perturbed_df(
        all_samples, *keys[filtering_mode], verbose=False
    )
    ## remove duplicate results just in case they were run twice
    dup_keys=["model_name", "task", "lang", "set_id", "var_id", "question"]
    duplicate_count = (samples.groupby(dup_keys, as_index=False).size()["size"] > 1).sum()
    print(f"N duplicates: {duplicate_count}")
    samples = samples.drop_duplicates(subset=dup_keys)
    metadata = {"filtering_mode": filtering_mode}
    #################### process results for each model ####################
    for model in model_names:
        df_filter_ = samples["model_name"] == model
        metadata["model_name"] = model
        model_df = samples[df_filter_]
        #################### process results for each model & task pair ####################
        for task in all_tasks:
            tmp = model_df[model_df["task"] == task]
            if len(tmp) == 0 and filtering_mode == "no_filter":
                warnings.warn(
                    f"Oh noo, check for model: {model}, task: {task} if it is run, the df is empty, skipping for now..."
                )
            if len(tmp) == 0:
                continue
            #################### extract canonical information ####################
            corr_canonical = model_df[model_df["is_canonical"] & model_df["set_id"].isin(
                tmp["set_id"].unique()
            )]
            canonical_count = len(corr_canonical)
            cor_canonical_task_name = corr_canonical["task"].unique()[0]
            task_group_name = cor_canonical_task_name.replace("_canonical", "")
            if len(tmp) == 0:
                pass
                print(f"Empty df for task {task} under filter: {filtering_mode}")
                continue
            category = tmp["final_category"].unique()
            task_pretty_name = tmp["task_pretty_name"].unique()[0]
            category = SUBCATEGORY_TO_CATEGORY.get(task_pretty_name, "")
            metadata.update(
                {
                    "task": task,
                    "task_pretty_name": task_pretty_name,
                    "canonical_count": canonical_count,
                    "num_samples": len(tmp),
                    "category": category,
                    "langs": ",".join(tmp["lang"].unique()),
                }
            )
            # compute accuracy metrics
            acc_norm = tmp["acc_norm"].mean()
            acc_norm_std = tmp["acc_norm"].std()
            acc = tmp["acc"].mean()
            acc_std = tmp["acc"].std()
            summary = {
                "acc_norm": tmp["acc_norm"].mean(),
                "acc_norm_std": tmp["acc_norm"].std(),
                "acc": tmp["acc"].mean(),
                "acc_std": tmp["acc"].std(),
                "canonical_task_name": cor_canonical_task_name,
                "canonical_acc": corr_canonical["acc"].mean(),
                "canonical_acc_norm": corr_canonical["acc_norm"].mean(),
                "group_name": task_group_name
            }
            all_summaries.append(metadata | summary)


all_summaries = pd.DataFrame.from_records(all_summaries)
all_summaries

N duplicates: 30604


,filtering_mode,model_name,task,task_pretty_name,canonical_count,num_samples,category,langs,acc_norm,acc_norm_std,acc,acc_std,canonical_task_name,canonical_acc,canonical_acc_norm
0,no_filter,Aya,tokenizer_robustness_completion_italian_canonical,Canonical,40,40,,ita_Latn,0.975000,0.158114,0.900000,0.303822,tokenizer_robustness_completion_italian_canonical,0.900000,0.975000
1,no_filter,Aya,tokenizer_robustness_completion_italian_orthog...,Orthographic errors,30,92,Script / Orthography,ita_Latn,0.847826,0.361158,0.760870,0.428890,tokenizer_robustness_completion_italian_canonical,0.866667,0.966667
2,no_filter,Aya,tokenizer_robustness_completion_italian_plausi...,Plausible diacritics errors,16,17,Script / Orthography,ita_Latn,0.882353,0.332106,0.941176,0.242536,tokenizer_robustness_completion_italian_canonical,0.937500,1.000000
3,no_filter,Aya,tokenizer_robustness_completion_italian_englis...,English keyboard,34,67,Script / Orthography,ita_Latn,0.820896,0.386334,0.791045,0.409631,tokenizer_robustness_completion_italian_canonical,0.882353,0.970588
4,no_filter,Aya,tokenizer_robustness_completion_italian_typogr...,Typographical errors,36,281,Noise,ita_Latn,0.715302,0.452075,0.750890,0.433269,tokenizer_robustness_completion_italian_canonical,0.888889,0.972222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1353,no_filter,mBERT,tokenizer_robustness_completion_stem_upside_do...,Upside Down/Rotated,16,17,Structural Text Elements,eng_Latn,0.176471,0.392953,0.235294,0.437237,tokenizer_robustness_completion_stem_canonical,0.875000,0.750000
1354,no_filter,mBERT,tokenizer_robustness_completion_stem_spelled_out,Spelled out,17,25,Mathematical & Scientific Notation,eng_Latn,0.680000,0.476095,0.680000,0.476095,tokenizer_robustness_completion_stem_canonical,0.823529,0.882353
1355,no_filter,mBERT,tokenizer_robustness_completion_stem_unusual_f...,Unusual formatting,13,16,Structural Text Elements,eng_Latn,0.687500,0.478714,0.437500,0.512348,tokenizer_robustness_completion_stem_canonical,0.846154,1.000000
1356,no_filter,mBERT,tokenizer_robustness_completion_stem_latex,LaTeX,23,68,Mathematical & Scientific Notation,eng_Latn,0.691176,0.465443,0.500000,0.503718,tokenizer_robustness_completion_stem_canonical,0.826087,0.913043


In [172]:
# all_summaries["task"].unique()
print(
    all_summaries[
        all_summaries["task_pretty_name"] == "Fullwidth Characters"
    ].to_markdown(index=False)
)


| filtering_mode   | model_name   | task                                                      | task_pretty_name     |   canonical_count |   num_samples | category                 | langs    |   acc_norm |   acc_norm_std |      acc |   acc_std | canonical_task_name                            |   canonical_acc |   canonical_acc_norm |
|:-----------------|:-------------|:----------------------------------------------------------|:---------------------|------------------:|--------------:|:-------------------------|:---------|-----------:|---------------:|---------:|----------:|:-----------------------------------------------|----------------:|---------------------:|
| no_filter        | Aya          | tokenizer_robustness_completion_stem_fullwidth_characters | Fullwidth Characters |                17 |            17 | Structural Text Elements | eng_Latn |   0.294118 |       0.469668 | 0.352941 |  0.492592 | tokenizer_robustness_completion_stem_canonical |        0.882353 |             0.8